In [1]:
from pathlib import Path
import numpy as np
import pandas as pd


PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent


CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ibm_tabformer"
    / "card_transaction.v1.csv"
).resolve()


RISK_COLUMNS = [
    "User",
    "Card",
    "Year",
    "Amount",
    "Use Chip",
    "Merchant Name",
    "Merchant State",
    "Zip",
    "MCC",
    "Errors?",
    "Is Fraud?",
]


dtype_map = {
    "User": "int32",
    "Card": "int16",
    "Year": "int16",
    "Use Chip": "category",
    "Merchant Name": "int64",
    "Merchant State": "category",
    "Zip": "float32",
    "MCC": "int16",
    "Errors?": "category",
    "Is Fraud?": "category",
}


df = pd.read_csv(
    CSV_PATH,
    usecols=RISK_COLUMNS,
    dtype=dtype_map,
)


print("Shape:", df.shape)

print("\nDtypes:")
print(df.dtypes.to_string())

print("\nMemory usage (MiB):")
print(
    round(
        df.memory_usage(
            deep=True
        ).sum() / 1024**2,
        2
    )
)

Shape: (24386900, 11)

Dtypes:
User                 int32
Card                 int16
Year                 int16
Amount                 str
Use Chip          category
Merchant Name        int64
Merchant State    category
Zip                float32
MCC                  int16
Errors?           category
Is Fraud?         category

Memory usage (MiB):
1906.31


In [2]:
df["_is_fraud"] = (
    df["Is Fraud?"]
    .eq("Yes")
    .astype("int8")
)


amount_numeric = pd.to_numeric(
    df["Amount"]
    .astype("string")
    .str.replace(
        "$",
        "",
        regex=False
    ),
    errors="coerce"
)


print(
    "Fraud values:",
    df["_is_fraud"]
    .value_counts()
    .sort_index()
    .to_dict()
)

print(
    "Amount parse failures:",
    int(amount_numeric.isna().sum())
)

Fraud values: {0: 24357143, 1: 29757}
Amount parse failures: 0


In [3]:
df["AmountNumeric"] = (
    amount_numeric
    .astype("float32")
)

del amount_numeric

df.drop(
    columns=["Amount"],
    inplace=True
)

print(
    "Memory usage after conversion (MiB):",
    round(
        df.memory_usage(
            deep=True
        ).sum() / 1024**2,
        2
    )
)

Memory usage after conversion (MiB): 744.25


In [4]:
year_risk = (
    df
    .groupby(
        "Year",
        observed=True
    )["_is_fraud"]
    .agg(
        total="size",
        fraud="sum"
    )
)

year_risk["non_fraud"] = (
    year_risk["total"]
    - year_risk["fraud"]
)

year_risk["fraud_rate_pct"] = (
    year_risk["fraud"]
    / year_risk["total"]
    * 100
)


print(
    year_risk.to_string()
)


total = len(df)

fraud = int(
    df["_is_fraud"].sum()
)

fraud_rate = (
    fraud / total * 100
)

majority_accuracy = (
    100 - fraud_rate
)


print(
    "\nOverall fraud:",
    f"{fraud:,}"
)

print(
    "Overall fraud rate:",
    f"{fraud_rate:.6f}%"
)

print(
    "Accuracy nếu luôn đoán NON-FRAUD:",
    f"{majority_accuracy:.6f}%"
)

        total  fraud  non_fraud  fraud_rate_pct
Year                                           
1991     1585      0       1585        0.000000
1992     5134      0       5134        0.000000
1993     8378      0       8378        0.000000
1994    14316      0      14316        0.000000
1995    20928      0      20928        0.000000
1996    29945     10      29935        0.033395
1997    49753     32      49721        0.064318
1998    78345     32      78313        0.040845
1999   118250     24     118226        0.020296
2000   177729    171     177558        0.096214
2001   257998    354     257644        0.137210
2002   350732    139     350593        0.039631
2003   466408    311     466097        0.066680
2004   597003    620     596383        0.103852
2005   746653    229     746424        0.030670
2006   908793   1118     907675        0.123020
2007  1064483   1881    1062602        0.176705
2008  1223460   3710    1219750        0.303238
2009  1355434   1140    1354294        0

In [5]:
has_error = (
    df["Errors?"]
    .notna()
)


def print_binary_mask_summary(
    mask,
    name_true,
    name_false
):
    for label, current_mask in [
        (name_false, ~mask),
        (name_true, mask),
    ]:
        total = int(
            current_mask.sum()
        )

        fraud = int(
            df.loc[
                current_mask,
                "_is_fraud"
            ].sum()
        )

        rate = (
            fraud / total * 100
            if total > 0
            else np.nan
        )

        print(
            label,
            "| rows:",
            f"{total:,}",
            "| fraud:",
            f"{fraud:,}",
            "| fraud rate:",
            f"{rate:.6f}%"
        )


print_binary_mask_summary(
    has_error,
    "HAS_ERROR",
    "NO_ERROR"
)

NO_ERROR | rows: 23,998,469 | fraud: 28,471 | fraud rate: 0.118637%
HAS_ERROR | rows: 388,431 | fraud: 1,286 | fraud rate: 0.331076%


In [6]:
error_risk = (
    df
    .groupby(
        "Errors?",
        observed=True,
        dropna=False
    )["_is_fraud"]
    .agg(
        total="size",
        fraud="sum"
    )
)

error_risk["fraud_rate_pct"] = (
    error_risk["fraud"]
    / error_risk["total"]
    * 100
)

error_risk = (
    error_risk
    .sort_values(
        [
            "total",
            "fraud"
        ],
        ascending=False
    )
)


print(
    error_risk.to_string()
)

                                                         total  fraud  fraud_rate_pct
Errors?                                                                              
NaN                                                   23998469  28471        0.118637
Insufficient Balance,                                   242783    396        0.163109
Bad PIN,                                                 58918    302        0.512577
Technical Glitch,                                        48157     63        0.130822
Bad Card Number,                                         13321    105        0.788229
Bad CVV,                                                 10740    280        2.607076
Bad Expiration,                                          10716    120        1.119821
Bad Zipcode,                                              2079      0        0.000000
Bad PIN,Insufficient Balance,                              581      7        1.204819
Insufficient Balance,Technical Glitch,                

In [7]:
location_missing = (
    df
    .groupby(
        "Use Chip",
        observed=True
    )
    .agg(
        total=("User", "size"),

        state_missing=(
            "Merchant State",
            lambda s: s.isna().sum()
        ),

        zip_missing=(
            "Zip",
            lambda s: s.isna().sum()
        ),
    )
)


location_missing[
    "state_missing_pct"
] = (
    location_missing[
        "state_missing"
    ]
    / location_missing["total"]
    * 100
)


location_missing[
    "zip_missing_pct"
] = (
    location_missing[
        "zip_missing"
    ]
    / location_missing["total"]
    * 100
)


print(
    location_missing.to_string()
)

                       total  state_missing  zip_missing  state_missing_pct  zip_missing_pct
Use Chip                                                                                    
Chip Transaction     6287598           7601        59122           0.120889         0.940295
Online Transaction   2713220        2713220      2713220         100.000000       100.000000
Swipe Transaction   15386082              0       105793           0.000000         0.687589


In [8]:
state_missing_mask = (
    df["Merchant State"].isna()
)

zip_missing_mask = (
    df["Zip"].isna()
)


print("\nMerchant State missing:")
print_binary_mask_summary(
    state_missing_mask,
    "STATE_MISSING",
    "STATE_PRESENT"
)


print("\nZip missing:")
print_binary_mask_summary(
    zip_missing_mask,
    "ZIP_MISSING",
    "ZIP_PRESENT"
)


Merchant State missing:
STATE_PRESENT | rows: 21,666,079 | fraud: 11,408 | fraud rate: 0.052654%
STATE_MISSING | rows: 2,720,821 | fraud: 18,349 | fraud rate: 0.674392%

Zip missing:
ZIP_PRESENT | rows: 21,508,765 | fraud: 4,905 | fraud rate: 0.022805%
ZIP_MISSING | rows: 2,878,135 | fraud: 24,852 | fraud rate: 0.863476%


In [9]:
negative_amount = (
    df["AmountNumeric"] < 0
)

zero_amount = (
    df["AmountNumeric"] == 0
)

positive_amount = (
    df["AmountNumeric"] > 0
)


print("NEGATIVE:")
print_binary_mask_summary(
    negative_amount,
    "NEGATIVE",
    "NOT_NEGATIVE"
)


print("\nZERO:")
print_binary_mask_summary(
    zero_amount,
    "ZERO",
    "NOT_ZERO"
)

negative_by_channel = (
    df.loc[
        negative_amount
    ]
    .groupby(
        "Use Chip",
        observed=True
    )
    .agg(
        rows=("User", "size"),
        fraud=("_is_fraud", "sum"),
    )
)

negative_by_channel[
    "fraud_rate_pct"
] = (
    negative_by_channel["fraud"]
    / negative_by_channel["rows"]
    * 100
)


print(
    negative_by_channel.to_string()
)

negative_sample = (
    df.loc[
        negative_amount,
        [
            "User",
            "Card",
            "Year",
            "AmountNumeric",
            "Use Chip",
            "Merchant Name",
            "Merchant State",
            "Zip",
            "MCC",
            "Errors?",
            "Is Fraud?",
        ]
    ]
    .sample(
        n=10,
        random_state=42
    )
)


print(
    negative_sample.to_string(
        index=False
    )
)

NEGATIVE:
NOT_NEGATIVE | rows: 23,142,217 | fraud: 28,632 | fraud rate: 0.123722%
NEGATIVE | rows: 1,244,683 | fraud: 1,125 | fraud rate: 0.090384%

ZERO:
NOT_ZERO | rows: 24,366,687 | fraud: 29,744 | fraud rate: 0.122068%
ZERO | rows: 20,213 | fraud: 13 | fraud rate: 0.064315%
                      rows  fraud  fraud_rate_pct
Use Chip                                         
Chip Transaction    342120     83        0.024260
Online Transaction   13499    896        6.637529
Swipe Transaction   889064    146        0.016422
 User  Card  Year  AmountNumeric          Use Chip        Merchant Name Merchant State     Zip  MCC Errors? Is Fraud?
  819     0  2013          -95.0 Swipe Transaction  1799189980464955940             TX 78210.0 5499     NaN        No
  341     1  2008          -63.0 Swipe Transaction   -85070450838390220             TX 75253.0 5541     NaN        No
 1669     0  2012          -57.0 Swipe Transaction -1288082279022882052             RI  2904.0 5499     NaN        No

In [10]:
def entity_risk_summary(
    data,
    group_cols,
    label
):
    stats = (
        data
        .groupby(
            group_cols,
            observed=True
        )["_is_fraud"]
        .agg(
            tx_count="size",
            fraud_count="sum"
        )
    )

    stats[
        "fraud_rate"
    ] = (
        stats["fraud_count"]
        / stats["tx_count"]
    )

    total_entities = len(stats)

    entities_with_fraud = int(
        (
            stats["fraud_count"] > 0
        ).sum()
    )

    entities_without_fraud = (
        total_entities
        - entities_with_fraud
    )

    fraud_only = int(
        (
            (
                stats["fraud_count"]
                == stats["tx_count"]
            )
            & (
                stats["tx_count"] > 0
            )
        ).sum()
    )

    print(
        f"\n=== {label} ==="
    )

    print(
        "Total entities:",
        f"{total_entities:,}"
    )

    print(
        "Entities with >= 1 fraud:",
        f"{entities_with_fraud:,}"
    )

    print(
        "Entities with no fraud:",
        f"{entities_without_fraud:,}"
    )

    print(
        "Entities where every transaction is fraud:",
        f"{fraud_only:,}"
    )

    print(
        "Max transaction count:",
        f"{int(stats['tx_count'].max()):,}"
    )

    print(
        "Max fraud count:",
        f"{int(stats['fraud_count'].max()):,}"
    )

    supported = stats[
        stats["tx_count"] >= 100
    ]

    if len(supported) > 0:
        print(
            "Max fraud rate among entities "
            "with >=100 transactions:",
            f"{supported['fraud_rate'].max() * 100:.6f}%"
        )

    return stats

user_stats = entity_risk_summary(
    df,
    ["User"],
    "USER"
)

card_stats = entity_risk_summary(
    df,
    [
        "User",
        "Card"
    ],
    "USER + CARD"
)

merchant_stats = entity_risk_summary(
    df,
    ["Merchant Name"],
    "MERCHANT"
)


=== USER ===
Total entities: 2,000
Entities with >= 1 fraud: 1,343
Entities with no fraud: 657
Entities where every transaction is fraud: 0
Max transaction count: 82,355
Max fraud count: 113
Max fraud rate among entities with >=100 transactions: 1.766980%

=== USER + CARD ===
Total entities: 6,139
Entities with >= 1 fraud: 2,752
Entities with no fraud: 3,387
Entities where every transaction is fraud: 0
Max transaction count: 70,008
Max fraud count: 57
Max fraud rate among entities with >=100 transactions: 15.267176%

=== MERCHANT ===
Total entities: 100,343
Entities with >= 1 fraud: 2,831
Entities with no fraud: 97,512
Entities where every transaction is fraud: 731
Max transaction count: 1,130,230
Max fraud count: 1,607
Max fraud rate among entities with >=100 transactions: 100.000000%


In [11]:
past_mask = (
    df["Year"] <= 2018
)

future_mask = (
    df["Year"] == 2019
)


def partition_summary(
    mask,
    label
):
    rows = int(
        mask.sum()
    )

    fraud = int(
        df.loc[
            mask,
            "_is_fraud"
        ].sum()
    )

    rate = (
        fraud / rows * 100
    )

    print(
        label,
        "| rows:",
        f"{rows:,}",
        "| fraud:",
        f"{fraud:,}",
        "| fraud rate:",
        f"{rate:.6f}%"
    )


partition_summary(
    past_mask,
    "PAST <= 2018"
)

partition_summary(
    future_mask,
    "FUTURE 2019"
)

past_users = (
    df.loc[
        past_mask,
        "User"
    ]
    .unique()
)

future_users = (
    df.loc[
        future_mask,
        "User"
    ]
    .unique()
)


past_merchants = (
    df.loc[
        past_mask,
        "Merchant Name"
    ]
    .unique()
)

future_merchants = (
    df.loc[
        future_mask,
        "Merchant Name"
    ]
    .unique()
)


CARD_BASE = (
    int(df["Card"].max())
    + 1
)


past_card_key = (
    df.loc[
        past_mask,
        "User"
    ].astype("int64")
    * CARD_BASE
    +
    df.loc[
        past_mask,
        "Card"
    ].astype("int64")
)


future_card_key = (
    df.loc[
        future_mask,
        "User"
    ].astype("int64")
    * CARD_BASE
    +
    df.loc[
        future_mask,
        "Card"
    ].astype("int64")
)


past_cards = (
    past_card_key.unique()
)

future_cards = (
    future_card_key.unique()
)


print("\n=== ENTITY OVERLAP ===")

print(
    "2019 users:",
    len(future_users)
)

print(
    "2019 users already seen in past:",
    int(
        np.isin(
            future_users,
            past_users
        ).sum()
    )
)


print(
    "\n2019 cards:",
    len(future_cards)
)

print(
    "2019 cards already seen in past:",
    int(
        np.isin(
            future_cards,
            past_cards
        ).sum()
    )
)


print(
    "\n2019 merchants:",
    len(future_merchants)
)

print(
    "2019 merchants already seen in past:",
    int(
        np.isin(
            future_merchants,
            past_merchants
        ).sum()
    )
)

future_user_is_new = (
    ~np.isin(
        df.loc[
            future_mask,
            "User"
        ].to_numpy(),
        past_users
    )
)


future_card_is_new = (
    ~np.isin(
        future_card_key.to_numpy(),
        past_cards
    )
)


future_merchant_is_new = (
    ~np.isin(
        df.loc[
            future_mask,
            "Merchant Name"
        ].to_numpy(),
        past_merchants
    )
)


future_rows = int(
    future_mask.sum()
)


print(
    "\n2019 transactions from NEW USER:",
    f"{int(future_user_is_new.sum()):,}",
    f"({future_user_is_new.mean() * 100:.6f}%)"
)


print(
    "2019 transactions from NEW CARD:",
    f"{int(future_card_is_new.sum()):,}",
    f"({future_card_is_new.mean() * 100:.6f}%)"
)


print(
    "2019 transactions from NEW MERCHANT:",
    f"{int(future_merchant_is_new.sum()):,}",
    f"({future_merchant_is_new.mean() * 100:.6f}%)"
)

del past_card_key
del future_card_key

PAST <= 2018 | rows: 22,326,462 | fraud: 27,670 | fraud rate: 0.123934%
FUTURE 2019 | rows: 1,723,938 | fraud: 2,087 | fraud rate: 0.121060%

=== ENTITY OVERLAP ===
2019 users: 1579
2019 users already seen in past: 1528

2019 cards: 4029
2019 cards already seen in past: 3883

2019 merchants: 34936
2019 merchants already seen in past: 32216

2019 transactions from NEW USER: 3,255 (0.188812%)
2019 transactions from NEW CARD: 18,544 (1.075677%)
2019 transactions from NEW MERCHANT: 4,905 (0.284523%)


In [12]:
raw_df = pd.read_csv(
    CSV_PATH
)

duplicate_members = (
    raw_df.loc[
        raw_df.duplicated(
            keep=False
        )
    ]
)


print(
    "Duplicate member rows:",
    len(duplicate_members)
)

print(
    "\nTarget among duplicate members:"
)

print(
    duplicate_members[
        "Is Fraud?"
    ]
    .value_counts(
        dropna=False
    )
    .to_string()
)


print(
    "\nLargest identical-row group:"
)

print(
    duplicate_members
    .groupby(
        list(raw_df.columns),
        dropna=False
    )
    .size()
    .max()
)

del raw_df
del duplicate_members

Duplicate member rows: 132

Target among duplicate members:
Is Fraud?
No    132

Largest identical-row group:
2


In [13]:
candidate_windows = [
    (2015, 2019),
    (2017, 2019),
    (2018, 2019),
    (2019, 2019),
]


print(
    "=== TEMPORAL WINDOW FEASIBILITY ==="
)


for start_year, end_year in candidate_windows:
    mask = (
        (df["Year"] >= start_year)
        &
        (df["Year"] <= end_year)
    )

    rows = int(
        mask.sum()
    )

    fraud = int(
        df.loc[
            mask,
            "_is_fraud"
        ].sum()
    )

    fraud_rate = (
        fraud / rows * 100
        if rows > 0
        else np.nan
    )

    users = (
        df.loc[
            mask,
            "User"
        ]
        .nunique()
    )

    cards = (
        df.loc[
            mask,
            [
                "User",
                "Card"
            ]
        ]
        .drop_duplicates()
        .shape[0]
    )


    print(
        f"\n{start_year}-{end_year}"
    )

    print(
        "Rows:",
        f"{rows:,}"
    )

    print(
        "Fraud:",
        f"{fraud:,}"
    )

    print(
        "Fraud rate:",
        f"{fraud_rate:.6f}%"
    )

    print(
        "Users:",
        f"{users:,}"
    )

    print(
        "User+Card:",
        f"{cards:,}"
    )

=== TEMPORAL WINDOW FEASIBILITY ===

2015-2019
Rows: 8,579,208
Fraud: 11,693
Fraud rate: 0.136295%
Users: 1,621
User+Card: 4,435

2017-2019
Rows: 5,168,913
Fraud: 4,833
Fraud rate: 0.093501%
Users: 1,598
User+Card: 4,236

2018-2019
Rows: 3,445,553
Fraud: 4,578
Fraud rate: 0.132867%
Users: 1,587
User+Card: 4,152

2019-2019
Rows: 1,723,938
Fraud: 2,087
Fraud rate: 0.121060%
Users: 1,579
User+Card: 4,029
